In [1]:
import pandas as pd
import numpy as np

In [2]:
file_path = r"C:\Users\HP\Desktop\Infotact Internship\week2_fused_dataset.csv"

In [3]:
df = pd.read_csv(file_path)

In [4]:
df.shape

(10000, 39)

In [5]:
import sys
print(sys.executable)

c:\Users\HP\Desktop\PredictX\predict_1\Scripts\python.exe


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, f1_score

In [7]:
internal_features = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',

    'Air temperature [K]_rolling_mean',
    'Air temperature [K]_rolling_std',
    'Air temperature [K]_rolling_var',

    'Process temperature [K]_rolling_mean',
    'Process temperature [K]_rolling_std',
    'Process temperature [K]_rolling_var',

    'Rotational speed [rpm]_rolling_mean',
    'Rotational speed [rpm]_rolling_std',
    'Rotational speed [rpm]_rolling_var',

    'Torque [Nm]_rolling_mean',
    'Torque [Nm]_rolling_std',
    'Torque [Nm]_rolling_var',

    'Tool wear [min]_rolling_mean',
    'Tool wear [min]_rolling_std',
    'Tool wear [min]_rolling_var'
]

In [8]:
context_features = internal_features + [
    'ambient_temp',
    'humidity',
    'factory_load',
    'ambient_gap',
    'load_stress',
    'heat_stress'
]

In [9]:
print("Internal Features :", len(internal_features))
print("Context Features :", len(context_features))

Internal Features : 20
Context Features : 26


In [10]:
target = 'Machine failure'

x_internal = df[internal_features]
x_context = df[context_features]

y = df[target]

In [11]:
print(x_internal.shape)
print(x_context.shape)
print(y.shape)

(10000, 20)
(10000, 26)
(10000,)


In [12]:
scorer = make_scorer(f1_score, average='macro')

In [13]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)

In [14]:
actual_score = cross_val_score(rf, x_internal, y, cv=5, scoring=scorer)

In [15]:
actual_score

array([0.74098461, 0.68172776, 0.39508847, 0.66914276, 0.77284193])

In [16]:
actual_score.mean()

np.float64(0.6519571080382367)

In [17]:
context_score = cross_val_score(rf, x_context, y, cv=5, scoring=scorer)

In [18]:
context_score

array([0.73591834, 0.68627998, 0.39675034, 0.7255301 , 0.76519809])

In [19]:
context_score.mean()

np.float64(0.6619353697753656)

In [20]:
print("\n----- Ablation Study Results -----")

print(
    f"Model A (Internal Only): "
    f"{actual_score.mean():.4f}"
)

print(
    f"Model B (Internal + Context): "
    f"{context_score.mean():.4f}"
)

improvement = (
    context_score.mean()
    - actual_score.mean()
)

print(
    f"Improvement: {improvement:.4f}"
)


----- Ablation Study Results -----
Model A (Internal Only): 0.6520
Model B (Internal + Context): 0.6619
Improvement: 0.0100


so the above describe there is a 1% improvement in prediction when we use some simple models with internal features alone vs internal and external features.